# BERT Email Classification Pipeline

This pipeline uses **BERT (DistilBERT)** models for both stages of email classification:
1. **Spam Detection**: Identifies if an email is Spam or Ham (Legitimate).
2. **Phishing Type Classification**: If Spam, categorizes the specific type of phishing attack.

BERT provides state-of-the-art performance by understanding the context and semantics of the email text.

In [6]:
import os
import torch
import joblib
import sys
import pandas as pd
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

# Add parent directory to path to import utils
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils.pipeline_utils import display_result, PHISHING_DESCRIPTIONS, test_emails

## Load Pre-trained BERT Models

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

models_dir = "../models"

# --- Load Stage 1: Spam BERT Model ---
spam_model_path = os.path.join(models_dir, "spam_bert_model")
spam_encoder_path = os.path.join(models_dir, "spam_bert_encoder.joblib")

print("Loading Spam BERT model...")
try:
    spam_tokenizer = DistilBertTokenizer.from_pretrained(spam_model_path)
    # Check number of labels from encoder to initialize model correctly if needed,
    # but from_pretrained usually handles config. Let's load encoder first.
    spam_encoder = joblib.load(spam_encoder_path)
    num_spam_labels = len(spam_encoder.classes_)
    
    spam_model = DistilBertForSequenceClassification.from_pretrained(
        spam_model_path, 
        num_labels=num_spam_labels
    )
    spam_model.to(device)
    spam_model.eval()
    print(f"Spam model loaded. Classes: {spam_encoder.classes_}")
except Exception as e:
    print(f"Error loading Spam BERT model: {e}")
    print("Ensure you have trained the spam BERT model using src/bert/train_spam_bert.py")


# --- Load Stage 2: Phishing BERT Model ---
phishing_model_path = os.path.join(models_dir, "phishing_bert_model")
phishing_encoder_path = os.path.join(models_dir, "phishing_bert_encoder.joblib")

print("Loading Phishing BERT model...")
try:
    phishing_tokenizer = DistilBertTokenizer.from_pretrained(phishing_model_path)
    phishing_encoder = joblib.load(phishing_encoder_path)
    num_phishing_labels = len(phishing_encoder.classes_)
    
    phishing_model = DistilBertForSequenceClassification.from_pretrained(
        phishing_model_path,
        num_labels=num_phishing_labels
    )
    phishing_model.to(device)
    phishing_model.eval()
    print(f"Phishing model loaded. Classes: {phishing_encoder.classes_}")
except Exception as e:
    print(f"Error loading Phishing BERT model: {e}")
    print("Ensure you have trained the phishing BERT model using src/bert/train_phishing_bert.py")

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RobertaTokenizer'. 
The class this function is called from is 'DistilBertTokenizer'.
You are using a model of type roberta to instantiate a model of type distilbert. This is not supported for all configurations of models and can yield errors.


Using device: cuda
Loading Spam BERT model...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at ../models\spam_bert_model and are newly initialized: ['classifier.bias', 'classifier.weight', 'distilbert.embeddings.LayerNorm.bias', 'distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.position_embeddings.weight', 'distilbert.embeddings.word_embeddings.weight', 'distilbert.transformer.layer.0.attention.k_lin.bias', 'distilbert.transformer.layer.0.attention.k_lin.weight', 'distilbert.transformer.layer.0.attention.out_lin.bias', 'distilbert.transformer.layer.0.attention.out_lin.weight', 'distilbert.transformer.layer.0.attention.q_lin.bias', 'distilbert.transformer.layer.0.attention.q_lin.weight', 'distilbert.transformer.layer.0.attention.v_lin.bias', 'distilbert.transformer.layer.0.attention.v_lin.weight', 'distilbert.transformer.layer.0.ffn.lin1.bias', 'distilbert.transformer.layer.0.ffn.lin1.weight', 'distilbert.transformer.layer.0.ffn.lin2.bias', 'distilbert.transformer.

Spam model loaded. Classes: ['ham' 'spam']
Loading Phishing BERT model...
Phishing model loaded. Classes: ['authority_scam' 'credential_harvesting' 'financial_scam'
 'generic_phishing' 'legitimate' 'romance_dating' 'social_engineering'
 'social_engineering_advanced' 'tech_support' 'threats' 'urgency']


## Define BERT Classification Functions

In [ ]:
def predict_bert(text, model, tokenizer, label_encoder, device, max_len=128):
    """
    Generic BERT prediction function.
    Returns: (predicted_label, confidence, all_probabilities)
    """
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=max_len,
        return_token_type_ids=False,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    pred_idx = probs.argmax()
    pred_label = label_encoder.inverse_transform([pred_idx])[0]
    confidence = probs[pred_idx]
    
    return pred_label, confidence, probs

def classify_email_bert(email_text):
    """
    Pipeline function for BERT models.
    """
    result = {
        'email_text': email_text[:200] + '...' if len(email_text) > 200 else email_text,
        'stage1_result': None,
        'stage1_confidence': None,
        'stage2_result': None,
        'stage2_confidence': None,
        'stage2_alternative': None,
        'stage2_alternative_confidence': None,
        'final_classification': None,
        'risk_level': None,
        'advice': None,
        'has_conflict': False,
        'uncertainty_warning': None
    }

    # Stage 1: Spam Detection
    spam_pred, spam_conf, spam_probs = predict_bert(
        email_text, spam_model, spam_tokenizer, spam_encoder, device
    )
    

    is_spam = (spam_pred.lower() == 'spam')
    result['stage1_result'] = 'Spam' if is_spam else 'Ham'
    result['stage1_confidence'] = spam_conf
    
    if not is_spam:
        result['final_classification'] = '✅ Legitimate Email (Ham)'
        result['risk_level'] = 'Low'
        result['advice'] = 'This email appears to be legitimate, but always verify sender addresses.'
        return result
        
    #Stage 2: Phishing Classification
    phishing_pred, phishing_conf, phishing_probs = predict_bert(
        email_text, phishing_model, phishing_tokenizer, phishing_encoder, device
    )
    
    # Get top 2 predictions
    sorted_indices = phishing_probs.argsort()[::-1]
    top_idx = sorted_indices[0]
    second_idx = sorted_indices[1] if len(sorted_indices) > 1 else None
    
    top_pred = phishing_encoder.inverse_transform([top_idx])[0]
    top_conf = phishing_probs[top_idx]
    
    second_pred = phishing_encoder.inverse_transform([second_idx])[0] if second_idx is not None else None
    second_conf = phishing_probs[second_idx] if second_idx is not None else None
    
    result['stage2_result'] = top_pred
    result['stage2_confidence'] = top_conf
    result['stage2_alternative'] = second_pred
    result['stage2_alternative_confidence'] = second_conf
    
    # --- Conflict Resolution ---
    # Check for conflict: Stage 1 says Spam, but Stage 2 says Legitimate
    is_conflict = (top_pred == 'legitimate')
    result['has_conflict'] = is_conflict
    
    if is_conflict:
        confidence_diff = abs(spam_conf - top_conf)
        
        if spam_conf > top_conf:
            # Trust Spam detection
            if second_pred and second_pred != 'legitimate':
                result['uncertainty_warning'] = (
                    f"⚠️ CONFLICTING PREDICTIONS: BERT spam detector flagged this as spam "
                    f"({spam_conf:.1%} confident), but phishing classifier suggested "
                    f"'legitimate' ({top_conf:.1%} confident). "
                    f"Showing next most likely phishing type instead."
                )
                info = PHISHING_DESCRIPTIONS.get(second_pred, {'label': second_pred, 'risk_level': 'High', 'advice': 'Be cautious.'})
                result['final_classification'] = f"⚠️ {info['label']} (Uncertain)"
                result['risk_level'] = 'Medium-High'
                result['advice'] = info['advice'] + " Note: There is some uncertainty in this classification."
            else:
                result['uncertainty_warning'] = (
                    f"⚠️ CONFLICTING PREDICTIONS: Spam detector says spam ({spam_conf:.1%}), "
                    f"but phishing classifier says legitimate ({top_conf:.1%}). "
                    f"Treating as suspicious due to spam detection."
                )
                result['final_classification'] = '⚠️ Suspicious Email (Unclassified Spam)'
                result['risk_level'] = 'Medium'
                result['advice'] = 'This email was flagged as spam but could not be categorized. Exercise caution.'
        else:
            # Trust Phishing (Legitimate) detection
            if confidence_diff > 0.2:
                 result['uncertainty_warning'] = (
                    f"⚠️ MIXED SIGNALS: Initially flagged as spam ({spam_conf:.1%}), "
                    f"but phishing analysis suggests legitimate ({top_conf:.1%}). "
                    f"Likely safe, but verify sender."
                )
                 result['final_classification'] = '⚠️ Likely Legitimate (Verify Sender)'
                 result['risk_level'] = 'Low-Medium'
                 result['advice'] = 'This email shows mixed signals. Double-check the sender address.'
            else:
                result['uncertainty_warning'] = (
                    f"⚠️ UNCERTAIN CLASSIFICATION: Spam detector ({spam_conf:.1%}) and "
                    f"phishing classifier ({top_conf:.1%}) disagree. "
                    f"Proceed with caution."
                )
                result['final_classification'] = '⚠️ Uncertain - Potential Spam'
                result['risk_level'] = 'Medium'
                result['advice'] = 'Our classifiers disagree on this email. Treat with caution.'
    else:
        # No conflict
        info = PHISHING_DESCRIPTIONS.get(top_pred, {'label': top_pred, 'risk_level': 'Unknown', 'advice': 'Be cautious.'})
        result['final_classification'] = info['label']
        result['risk_level'] = info['risk_level']
        result['advice'] = info['advice']
        
        if top_conf < 0.5:
            result['uncertainty_warning'] = (
                f"⚠️ LOW CONFIDENCE: The phishing type classification confidence is only "
                f"{top_conf:.1%}. The actual threat type may differ."
            )
            result['risk_level'] = 'Medium' if result['risk_level'] == 'Low' else result['risk_level']

    return result

## Test the BERT Pipeline

In [9]:
print("BERT EMAIL CLASSIFICATION RESULTS")

for i, email in enumerate(test_emails, 1):
    print(f"\n>>> Email #{i}")
    try:
        result = classify_email_bert(email)
        display_result(result)
    except Exception as e:
        print(f"Error classifying email: {e}")

BERT EMAIL CLASSIFICATION RESULTS

>>> Email #1
📧 EMAIL CLASSIFICATION RESULT

📝 Email Preview:
    Hi John,
    
    Just wanted to follow up on our meeting yesterday. I've attached the quarterly report 
    as discussed. Let me know if you have any questions.
    
    Best regards,
    Sarah

🔍 Stage 1 - Spam Detection:
    Result:     Spam
    Confidence: 62.5%

🎯 Stage 2 - Phishing Type Classification:
    Prediction: legitimate
    Confidence: 99.7%
    2nd Best:   social_engineering_advanced (0.1%)

🔀 CONFLICT DETECTED
    ⚠️ MIXED SIGNALS: Initially flagged as spam (62.5%), but phishing analysis suggests legitimate (99.7%). Likely safe, but verify sender.

📊 FINAL CLASSIFICATION:
    Result:     ⚠️ Likely Legitimate (Verify Sender)
    Risk Level: Low-Medium

💡 ADVICE:
    This email shows mixed signals. Double-check the sender address.


>>> Email #2
📧 EMAIL CLASSIFICATION RESULT

📝 Email Preview:
    URGENT: Your account has been compromised!
    
    We detected suspicious ac

## Interactive Classification

In [10]:
# Try your own email here
your_email = """
Subject: Urgent Account Update Required

Dear Customer,

We have noticed unusual activity on your account. Please click the link below to verify your identity.
If you do not verify within 24 hours, your account will be locked.
"""

try:
    result = classify_email_bert(your_email)
    display_result(result)
except Exception as e:
    print(f"Error: {e}")

📧 EMAIL CLASSIFICATION RESULT

📝 Email Preview:
    
Subject: Urgent Account Update Required

Dear Customer,

We have noticed unusual activity on your account. Please click the link below to verify your identity.
If you do not verify within 24 hours, y...

🔍 Stage 1 - Spam Detection:
    Result:     Spam
    Confidence: 59.2%

🎯 Stage 2 - Phishing Type Classification:
    Prediction: social_engineering
    Confidence: 89.6%
    2nd Best:   generic_phishing (2.8%)

📊 FINAL CLASSIFICATION:
    Result:     🎭 Social Engineering
    Risk Level: Medium

💡 ADVICE:
    Verify requests through official channels before taking action.

